# Persian Synthetic Instruct — QLoRA Fine-tuning
**Dataset:** [Heydaritoday/Persian-Synthetic-Instruct](https://huggingface.co/datasets/Heydaritoday/Persian-Synthetic-Instruct)  
**Model:** Qwen2.5-1.5B-Instruct  
**Method:** QLoRA with Unsloth


In [ ]:
# Step 1: Install dependencies
!pip install unsloth datasets -q
!pip install --upgrade --no-cache-dir unsloth -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [ ]:
# Step 2: Load model with Unsloth (4x faster, half VRAM)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# Step 3: Load dataset from Hugging Face
from datasets import load_dataset
import json

# Load all JSONL files from the dataset
dataset = load_dataset(
    "Heydaritoday/Persian-Synthetic-Instruct",
    split="train"
)

print(f"Total samples: {len(dataset)}")
print(dataset[0])

In [ ]:
# Step 4: Format dataset
def format_sample(sample):
    instruction = sample["instruction"]
    output = sample["output"]
    input_text = sample.get("input", "").strip()
    user_content = f"{instruction}\n\n{input_text}" if input_text else instruction
    text = f"<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
    return {"text": text}

formatted = dataset.map(format_sample)
split = formatted.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

In [ ]:
# Step 5: Train
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="finetuned_model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_steps=100,
    evaluation_strategy="steps",
    eval_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    optim="adamw_8bit",
    warmup_ratio=0.05,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
)

trainer.train()

In [ ]:
# Step 6: Push to Hugging Face
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Push model and tokenizer
model.push_to_hub("Heydaritoday/Persian-Qwen2.5-1.5B-Instruct")
tokenizer.push_to_hub("Heydaritoday/Persian-Qwen2.5-1.5B-Instruct")
print("Model pushed to Hugging Face!")

In [ ]:
# Step 7: Quick benchmark
FastLanguageModel.for_inference(model)

questions = [
    "تفاوت SSD و HDD چیه؟",
    "چطور می‌تونم استرس رو کاهش بدم؟",
    "هوش مصنوعی چیه و چطور کار می‌کنه؟",
]

for q in questions:
    prompt = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)